In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 27. Flash Attention과 메모리 최적화

> **학습 목표**
> - Flash Attention이 메모리 계층(GPU SRAM vs HBM)을 어떻게 활용하는지 이해한다
> - IO 복잡도 개념과 타일링(tiling)을 설명한다
> - 메모리 절약 효과를 실험으로 확인한다

## 27.1 GPU 메모리 계층

GPU에는 메모리 계층이 있다:
- **SRAM** (온칩, 매우 빠름, 작음 ~20MB)
- **HBM** (High Bandwidth 메모리, 보통 "GPU 메모리"라 부름, ~40-80GB)
- **DRAM** (CPU 메모리, 느림, 큼)

HBM ↔ SRAM 전송이 병목. 어텐션은 $n \times n$ 행렬을 HBM에 쓰고 읽음 → 느림.

## 27.2 표준 어텐션의 메모리 병목

$$S = QK^\top \in \mathbb{R}^{n \times n}$$  (HBM에 쓰기)
$$P = \mathrm{softmax}(S) \in \mathbb{R}^{n \times n}$$  (HBM 읽고 쓰기)
$$O = PV \in \mathbb{R}^{n \times d}$$  (HBM 읽고 쓰기)

- HBM 접근: $O(n^2)$ (대규모)
- SRAM 활용: 거의 없음

## 27.3 Flash Attention — 타일링과 Online Softmax

**핵심 아이디어**: $n \times n$ 행렬을 메모리에 두지 말고, 블록 단위로 SRAM에서 처리.

### Online Softmax
소프트맥스를 한 번에 계산할 필요 없이, 누적 $\max$와 $\sum$으로 점진적 계산:
$$m_i^{(j)} = \max(m_i^{(j-1)}, \max_j S_{ij})$$
$$\ell_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} \ell_i^{(j-1)} + \sum_j e^{S_{ij} - m_i^{(j)}}$$

### Tiling
$Q, K, V$를 블록으로 나누어 SRAM에 올리고, 부분 어텐션 계산 후 결과만 HBM에 쓰기.

**IO 복잡도**:
- 표준: $O(n^2)$ HBM 접근
- Flash: $O(n^2 d / M)$ where $M$ = SRAM 크기 → 훨씬 적음

### 역전파 재계산
역전파 시 $S, P$를 저장하지 않고 재계산 (메모리 $O(n)$ vs $O(n^2)$).


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 표준 어텐션
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-1, -2) / (d_k ** 0.5)  # HBM 쓰기
    P = F.softmax(S, dim=-1)  # HBM 읽기/쓰기
    O = P @ V  # HBM 읽기/쓰기
    return O

# Flash Attention 흉내 (간소화, online softmax)
def flash_attention_naive(Q, K, V, block_size=64):
    """Flash attention의 타일링 아이디어를 간소화한 버전."""
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)
    # Q를 Block으로, K/V는 전체 (실제 Flash는 둘 다 Block)
    for i in range(0, N, block_size):
        Q_block = Q[:, :, i:i+block_size, :]  # (B, H, bs, D)
        S_block = Q_block @ K.transpose(-1, -2) / (D ** 0.5)  # (B, H, bs, N)
        P_block = F.softmax(S_block, dim=-1)
        O[:, :, i:i+block_size, :] = P_block @ V
    return O

# 동일성 확인
torch.manual_seed(0)
B, H, N, D = 1, 2, 128, 32
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O_std = standard_attention(Q, K, V)
O_flash = flash_attention_naive(Q, K, V, block_size=32)
print(f"표준 vs Flash 최대 오차: {(O_std - O_flash).abs().max():.2e}")
print("=> 결과는 동일. Flash는 메모리 효율만 개선.")


## 27.4 PyTorch SDPA 백엔드

PyTorch의 `scaled_dot_product_attention`은 자동으로 최적 백엔드 선택:
- `flash`: Flash Attention v2 (GPU)
- `mem_efficient`: 메모리-Efficient Attention (GPU/CPU)
- `math`: 표준 구현


In [ ]:
# SDPA 백엔드 비교
import time

# 큰 시퀀스로 메모리 효과 확인
def bench_attention(n, d=64, n_heads=8, device='cpu'):
    """Attention Time과 메모리 Measurement."""
    Q = torch.randn(1, n_heads, n, d, device=device)
    K = torch.randn(1, n_heads, n, d, device=device)
    V = torch.randn(1, n_heads, n, d, device=device)

    # Standard 구현
    def std_attn():
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

    # SDPA
    def sdpa_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    # Time Measurement
    for _ in range(2): std_attn()  # warmup
    t0 = time.perf_counter()
    for _ in range(3):
        std_attn()
    t_std = (time.perf_counter() - t0) / 3 * 1000

    for _ in range(2): sdpa_attn()
    t0 = time.perf_counter()
    for _ in range(3):
        sdpa_attn()
    t_sdpa = (time.perf_counter() - t0) / 3 * 1000

    return t_std, t_sdpa

print(f"{'n':>6} | {'Standard (ms)':>15} | {'SDPA (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n in [256, 512, 1024, 2048]:
    t_std, t_sdpa = bench_attention(n, device='cpu')
    print(f"{n:>6} | {t_std:>15.3f} | {t_sdpa:>12.3f} | {t_std/t_sdpa:>9.2f}x")


## 27.5 메모리 절약 분석

표준 어텐션은 $n \times n$ 어텐션 행렬 저장 → $O(n^2)$ 메모리.

Flash Attention은 어텐션 행렬 저장 안 함 → $O(n)$ 메모리.

예: $n = 8192, h = 32, d_k = 128, B = 1$:
- 표준: $B \cdot h \cdot n^2 \cdot 4 = 8$ GB (FP32)
- Flash: $O(n)$ → 약 100MB


In [ ]:
# 메모리 분석
def attention_memory_gb(n, n_heads, d_k, batch=1, dtype_bytes=4):
    """Standard Attention의 Attention 행렬 메모리."""
    return batch * n_heads * n * n * dtype_bytes / (1024**3)

print(f"Attention 행렬 메모리 (FP32):")
print(f"{'n':>6} | {'메모리 (GB)':>12} | {'Flash (Estimation)':>14}")
print("-" * 40)
for n in [1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(n, n_heads=32, d_k=128)
    flash_mem = n * 32 * 128 * 4 / (1024**3)  # 대략 O(n)
    print(f"{n:>6} | {mem:>12.3f} | {flash_mem:>14.4f}")
print("\n=> n=8192에서 Standard은 8GB, Flash는 수십 MB. Difference 압도적.")


## 27.6 Flash Attention v2/v3

- **v1** (Dao et al., 2022): 타일링, online softmax
- **v2** (2023): 워크로드 밸런싱 개선, 역전파 최적화
- **v3** (2024): Hopper 아키텍처 (H100) 최적화, 비동기 처리

## 27.7 Ring Attention — 다중 GPU 분산

$Q, K, V$를 GPU에 분산, GPU 간 통신으로 전역 어텐션.
- 단일 GPU 메모리 한계 돌파
- 1M+ 토큰 컨텍스트 가능

## 27.8 핵심 정리

| 기법 | IO 복잡도 | 메모리 | 속도 |
|---|---|---|---|
| 표준 어텐션 | $O(n^2)$ | $O(n^2)$ | 느림 |
| Flash Attention v2 | $O(n^2 d/M)$ | $O(n)$ | 빠름 |
| Flash Attention v3 | 최적화 | $O(n)$ | H100에서 최고 |
| Ring Attention | 분산 | $O(n/k)$ (k GPU) | 매우 긴 컨텍스트 |

## 연습문제

1. PyTorch SDPA의 백엔드를 `math`, `mem_efficient`로 명시적 설정하고 비교하라.
2. 시퀀스 길이 1024, 4096, 16384에서 표준 vs SDPA 시간과 메모리를 비교하라.
3. Online Softmax를 직접 구현하고 표준 소프트맥스와 결과가 같음을 보이라.
4. Flash Attention이 역전파 메모리를 어떻게 절약하는지 설명하라.
5. Ring Attention이 다중 GPU에서 어떻게 동작하는지 설명하라.

> 해설: `solutions/ch27_solutions.ipynb`
